# Serving LLMs Efficiently with vLLM - Part II 

In [ ]:
         1         2         3         4         5         6         7  
#23456789012345678901234567890123456789012345678901234567890123456789012

vLLM brings in the inference optimizations we've been learning about like continuous batching so you're not wasting compute waiting for the longest request, pagedattention for managing the KV cache in fixed-size blocks with little memory waste, and prefix caching to reuse KV entries when requests share the same system prompt. Plus, it natively supports quantized models like the one we just built, however for today since this environment is on CPU we'll just be using the base Qwen model. 

## Step 1: Start Your vLLM Server

We've already pre-warmed a vLLM server for you to use with Qwen3 .6B.

To server your model with vLLM, you'd need to run this command in the terminal. 

```bash
vllm serve Qwen/Qwen3-0.6B --dtype=bfloat16 --max-model-len 4096
```

Now in this learning environment, we've already pre-warmed a vLLM server for you to use with Qwen3 .6B, so the server is already running and we'll use the code cells in this notebook to interact with it. Before that, let's go through what each piece in this command means.

- **`vllm serve`** — launches vLLM's built-in inference server. It loads the model weights into the engine (with PagedAttention, continuous batching, and prefix caching enabled by default) and exposes it over HTTP on port 8000.
- **`Qwen/Qwen3-0.6B`** — the model identifier on the [Hugging Face Hub](https://huggingface.co/Qwen/Qwen3-0.6B). On first run, vLLM downloads the weights, tokenizer, and config from Hugging Face into the local cache (`~/.cache/huggingface/hub`), then loads them into memory. Subsequent runs reuse the cached files.
- **`--dtype=bfloat16`** — loads the weights in BF16 precision. 
- **`--max-model-len 4096`** — caps the context window (prompt + generation) at 4096 tokens. vLLM uses this to size the KV cache block pool up front, so setting it sensibly avoids reserving memory you'll never use.

vLLM wraps the model in an **OpenAI-compatible HTTP API**. It implements the same routes the OpenAI SDK calls — `/v1/models`, `/v1/chat/completions`, `/v1/completions`, `/v1/embeddings` — with the same request and response shapes.

So here let's check if it's running. 

VLLM-URl is the address of the local server. What we're doing here is that we're sending a GET request to this endpoint http://localhost:8000/v1/models (vLLM's OpenAI-compatible "list models" endpoint) every 5 seconds until it gets a 200 response, meaning the server has finished loading weights. From that response it reads data[0].id into MODEL, which every later request reuses. If nothing answers within 5 minutes, it raises with the exact vllm serve command to run.

In [1]:
!pip install -q openai requests

import time, requests, json, os, math, sys
from openai import OpenAI

# ─── Sandbox connection ───────────────────────────────────────────────────
# Paste your token here (get it with: oc whoami -t)
TOKEN = os.environ.get("OPENSHIFT_TOKEN") or "sha256~u5npf30Rzws7Au2z2l01e3t_T1LW1ijR-oOz2LgIoH0"

VLLM_URL = os.environ.get("VLLM_URL",
    "https://isvc-qwen3-8b-fp8-predictor.sandbox-shared-models.svc.cluster.local:8443")
# ──────────────────────────────────────────────────────────────────

_AUTH_HEADERS = {"Authorization": f"Bearer {TOKEN}"}
client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key=TOKEN)
os.makedirs("outputs", exist_ok=True)

r = requests.get(f"{VLLM_URL}/v1/models", headers=_AUTH_HEADERS, timeout=10)
if r.status_code == 401:
    raise RuntimeError("Token expired — run `oc whoami -t` and paste a fresh one above.")
r.raise_for_status()
MODEL = r.json()["data"][0]["id"]
print(f"Connected — model: {MODEL}")

## Your First Local LLM Request

Now, let's send our first request. Since vLLM exposes a OpenAI-compatible API,  we can the standrd `openai` Python client, point it at `http://localhost:8000/v1` and then use it exactly as if we were calling OpenAI's hosted API. Same client code, same request format, just a different `base_url` — which is what makes it easy to prototype against a hosted model and then swap in a self-hosted one without rewriting your application

In [5]:
# client already created in setup cell above

Let's ask the model if it knows about PagedAttention, and set some sampling parameters. You'll notice that Qwen3 is a thinking model, but we'll turn that off to keep responses short, and enable it later for you to see.

In [6]:
start = time.time()
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", 
               "content": "What is PagedAttention in one sentence?"}],
    max_tokens=80,
    temperature=0.7,
    top_p=0.8,
    extra_body={"top_k": 20, 
                "chat_template_kwargs": {"enable_thinking": False}},
)
elapsed = time.time() - start

print(f"Response ({elapsed:.2f}s, {resp.usage.completion_tokens} tokens):")
print(resp.choices[0].message.content)
print(f"\nUsage: {resp.usage.prompt_tokens} prompt + "
      f"{resp.usage.completion_tokens} completion = {
      resp.usage.total_tokens} total")

Response (2.66s, 31 tokens):
PagedAttention is a technique used to improve the efficiency of attention mechanisms by dividing the attention space into smaller blocks and applying attention to each block independently.

Usage: 21 prompt + 31 completion = 52 total


## Exploring Model Behavior

Beyond just getting answers, running vLLM ourselves lets us look inside the model's decision-making. 

For example, logprobs. When we ask "the capital of france is" with temperature of 0 to get deterministic output, we see the model's confidence for every token it generates (plus alternatives it considered). For Paris, you'll see very high probability, meaning the model is confident. This is useful for understanding when a model is sure, versus when it's guessing

In [4]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "The capital of France is"}],
    max_tokens=15,
    temperature=0.0,
    logprobs=True,
    top_logprobs=5,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print(f"Response: {resp.choices[0].message.content.strip()}\n")
print("Token-by-token probabilities:\n")

for tok in resp.choices[0].logprobs.content[:8]:
    print(f"  Chosen: '{tok.token}'  (logprob {tok.logprob:.2f})")
    if tok.top_logprobs:
        for alt in tok.top_logprobs[:5]:
            pct = math.exp(alt.logprob) * 100
            bar = "\u2588" * min(20, max(1, int(pct / 5)))
            print(f"    {pct:5.1f}%  {bar}  '{alt.token}'")
    print()

Response: The capital of France is **Paris**.

Token-by-token probabilities:

  Chosen: 'The'  (logprob -0.00)
     99.9%  ███████████████████  'The'
      0.1%  █  'France'
      0.0%  █  'Capital'
      0.0%  █  'Paris'
      0.0%  █  'Le'

  Chosen: ' capital'  (logprob -0.00)
    100.0%  ███████████████████  ' capital'
      0.0%  █  ' Capital'
      0.0%  █  ' **'
      0.0%  █  ' French'
      0.0%  █  '**'

  Chosen: ' of'  (logprob -0.00)
    100.0%  ███████████████████  ' of'
      0.0%  █  ' city'
      0.0%  █  ' is'
      0.0%  █  ' ('
      0.0%  █  ' and'

  Chosen: ' France'  (logprob -0.00)
    100.0%  ███████████████████  ' France'
      0.0%  █  ' **'
      0.0%  █  ' the'
      0.0%  █  'France'
      0.0%  █  ' Europe'

  Chosen: ' is'  (logprob -0.00)
    100.0%  ███████████████████  ' is'
      0.0%  █  ' in'
      0.0%  █  ','
      0.0%  █  ' was'
      0.0%  █  ' **'

  Chosen: ' **'  (logprob -0.09)
     91.2%  ██████████████████  ' **'
      7.5%  █  ' Paris'

Now, temperature, we'll send the same prompt at three different settings, 0, .7, and 1.5. Temperature zero is deterministic, you'll most likely get the same tokens every time. 

.7 adds some randomness, and 1.5 makes the output more creative. Compare these outputs, you'll see the model get progressively more varied. This is how you control the predictibabilty between "i want it this way every time" and "I'm okay with diversity or creativity from the model", which are two common use cases.

In [5]:
prompt = "Write a one-sentence description of what vLLM does."

for temp in [0.0, 0.7, 1.5]:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=60,
        temperature=temp,
        seed=42,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    print(f"temp={temp}:  {resp.choices[0].message.content.strip()}\n")

temp=0.0:  VLLM is a large-scale language model that enables efficient and scalable training and inference of large language models.

temp=0.7:  VLLM is a library that enables large-scale model training and inference through efficient parallel processing.

temp=1.5:  VLLM is a model-efficient large language modeling framework designed for efficient, scalable processing of text sequences.



## Observing vLLM Under the Hood

Now, let's look at whats happening inside vLLM. It exposes a Prometheus-compatible (a specific format that's easy to scrape)/metrics endpoint, which we can scrape to see how many requests are running or waiting, the KV cache being used, and cumulative token counts.

Key metrics to watch:

- **`num_requests_running / waiting`** — how many requests are active vs queued
- **`gpu_cache_usage_perc`** (or `cpu_cache_usage_perc`) — KV cache memory pressure
- **`prompt_tokens_total / generation_tokens_total`** — cumulative token counts

This cell defines a helper that parses the metrics endpoint and prints the key stats. If you're running vLLM in the terminal, it also logs throughput and cache stats in realtime.

In [3]:
def get_vllm_metrics(base_url=VLLM_URL):
    """Scrape vLLM Prometheus /metrics and return {name: value}."""
    r = requests.get(f"{base_url}/metrics", headers=_AUTH_HEADERS)
    metrics = {}
    for line in r.text.split("\n"):
        if line.startswith("#") or not line.strip():
            continue
        name = line.split("{")[0].split()[0]
        try:
            metrics[name] = float(line.split()[-1])
        except (ValueError, IndexError):
            continue
    return metrics

metrics = get_vllm_metrics()
print("Current vLLM Metrics:")
for key in ["vllm:num_requests_running", "vllm:num_requests_waiting",
            "vllm:gpu_cache_usage_perc", "vllm:cpu_cache_usage_perc",
            "vllm:prompt_tokens_total", "vllm:generation_tokens_total"]:
    if key in metrics:
        print(f"  {key.replace('vllm:', '')}: {metrics[key]:g}")

with open("outputs/metrics_snapshot.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nFull metrics saved to outputs/metrics_snapshot.json")

Current vLLM Metrics:
  num_requests_running: 0
  num_requests_waiting: 0
  prompt_tokens_total: 0
  generation_tokens_total: 0

Full metrics saved to outputs/metrics_snapshot.json


## Continuous Batching in Action

So, this is where things get fun. We're going to send 5 requests at the same time, and watch vLLM handle them. 

We fire them off concurrently, and while they're in flight, we scrape the metrics to see how many requests and running versus waiting. 

On a GPU, you'd see multiple requests running simulatenously, that's vLLM's continuous batching processing them together, producing tokens for every request in the batch at each step, and when one finishes, it's slot is immediately filled.

**I was able to see 5 requests running concurrently**
On CPU, requests will queue and run sequentially, but the scheduler is still handling them. One thing to watch is the total time, it's faster than running them one by one because the scheduler is managing them efficetively. 

In [7]:
import concurrent.futures

prompts = [
    "What is quantization?",
    "Explain KV caching briefly.",
    "What is continuous batching?",
    "Why is LLM inference memory-bound?",
    "What is PagedAttention?",
]

def _ask(prompt):
    return client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=60, temperature=0.7,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )

before = get_vllm_metrics()
print(f"Sending {len(prompts)} concurrent requests...\n")
start = time.time()

with concurrent.futures.ThreadPoolExecutor(
    max_workers=len(prompts)) as pool:
    futures = {pool.submit(_ask, p): p for p in prompts}
    time.sleep(0.5)
    during = get_vllm_metrics()
    running = during.get("vllm:num_requests_running", "--")
    waiting = during.get("vllm:num_requests_waiting", "--")
    print(f"  [mid-flight]  running: {running}  |  waiting: {waiting}")

    for f in concurrent.futures.as_completed(futures):
        resp = f.result()
        print(f"  done: \"{futures[f][:40]}\" -> {resp.usage.completion_tokens} tokens")

elapsed = time.time() - start
after = get_vllm_metrics()
tokens = after.get("vllm:generation_tokens_total", 0) - before.get(
    "vllm:generation_tokens_total", 0)

print(f"\nAll {len(prompts)} completed in {elapsed:.2f}s")
if tokens > 0:
    print(f"Tokens generated: {tokens:g}  |  ~{tokens / elapsed:.1f} tokens/s")

Sending 5 concurrent requests...

  [mid-flight]  running: 5.0  |  waiting: 0.0
  done: "What is quantization?" -> 60 tokens
  done: "What is PagedAttention?" -> 60 tokens
  done: "What is continuous batching?" -> 60 tokens
  done: "Explain KV caching briefly." -> 60 tokens
  done: "Why is LLM inference memory-bound?" -> 60 tokens

All 5 completed in 8.35s
Tokens generated: 300  |  ~35.9 tokens/s


### What Just Happened

PagedAttention is what makes this work at scale. The KV cache generated is divided into fixed-size blocks that can go anywhere in memory. When a request is done, it's blocks are immediately available for reuse, and space isn't wasted.

## Prefix Caching

Let's see prefix caching. Many applications use the same system prompt across every request, and without this prefix caching, vLLM would recompute the KV cache for that shared prefix every time.

So, we setup a system prompt, and send 5 different questions with the same system prompt. The first request pays the full prefill cost, but after that, vLLM recognizes the shared prefix and skips recomputing it.

We track the prefix_cache_queries metric before and after, and you'll see it increase after each request, confirming that vLLM is checking and reusing the cache. With these short prompts, time savings aren't very visible, but in production with thousands of instructions or few show examples, this eliminates a huge amount of compute.

In [9]:
SYSTEM_PROMPT = (
    "You are a helpful AI teaching assistant for a course on "
    "LLM optimization. You specialize in explaining concepts like "
    "quantization, inference optimization, and model serving. Keep "
    "answers concise -- one or two sentences."
)

questions = [
    "What is weight quantization?",
    "How does vLLM handle memory?",
    "What is continuous batching?",
    "Why use prefix caching?",
    "What is GPTQ?",
]

before = get_vllm_metrics()
timings = []
tok_counts = []

print("Sending 5 requests with the SAME system prompt...\n")
for i, q in enumerate(questions):
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q},
        ],
        max_tokens=60, temperature=0.7,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    dt = time.time() - t0
    timings.append(dt)
    tok_counts.append(resp.usage.completion_tokens)
    ms_per_tok = dt / resp.usage.completion_tokens * 1000
    print(f"  [{i+1}] {q:<35} {dt:.2f}s  ({resp.usage.completion_tokens} tok, {ms_per_tok:.0f} ms/tok)")

after = get_vllm_metrics()

prefix_before = before.get("vllm:prefix_cache_queries_total", 0)
prefix_after = after.get("vllm:prefix_cache_queries_total", 0)

print(f"\nPrefix cache queries: {prefix_before:g} -> {prefix_after:g}  (+{prefix_after - prefix_before:g})")

cache_keys = [k for k in after if "prefix" in k.lower() 
              or "cache_hit" in k.lower()]
for k in sorted(cache_keys):
    b, a = before.get(k, 0), after.get(k, 0)
    if a != b and k != "vllm:prefix_cache_queries_total":
        print(f"  {k}: {b:g} -> {a:g}")

print("\n The increasing prefix_cache_queries count confirms vLLM is ")
print("checking and reusing cached KV blocks for the shared system prompt.")

Sending 5 requests with the SAME system prompt...

  [1] What is weight quantization?        4.54s  (42 tok, 108 ms/tok)
  [2] How does vLLM handle memory?        2.39s  (22 tok, 109 ms/tok)
  [3] What is continuous batching?        3.80s  (41 tok, 93 ms/tok)
  [4] Why use prefix caching?             2.50s  (31 tok, 81 ms/tok)
  [5] What is GPTQ?                       4.60s  (45 tok, 102 ms/tok)

Prefix cache queries: 278 -> 593  (+315)

 The increasing prefix_cache_queries count confirms vLLM is 
checking and reusing cached KV blocks for the shared system prompt.


---
## (Optional) KV Cache Size Visualization for Qwen3-0.6B

This optional section let's you calculate exactly how much memory the KV cache consumes. For Qwen3 .6B with 28 layers, 8 KV heads, and 128-dimentional head size in bfloat16, we compute the per-token cost and scale it up. You'll see that a single token is relatively small, but a 4096 token context adds up. And, 10 concurrent users at full context. This is the challenge of managing GPU memory, and why pagedattention matters, it manages this memory efficiently instead of pre-allocating for worst case.

In [10]:
num_layers = 28
num_kv_heads = 8     # GQA: 16 Q heads, 8 KV heads
head_dim = 128
dtype_bytes = 2      # BF16

per_token = 2 * num_layers * num_kv_heads * head_dim * dtype_bytes

print(f"KV Cache -- Qwen3-0.6B")
print(f"Per token: 2 x {num_layers} x {num_kv_heads} x {head_dim} x {dtype_bytes}"
      f" = {per_token:,} bytes ({per_token // 1024} KB)\n")
print(f"  {'Context':>8}  {'KV Cache':>10}")
print(f"  {'-'*8}  {'-'*10}")
for ctx in [1, 64, 256, 1024, 4096]:
    size = per_token * ctx
    label = f"{size / 1024:.0f} KB" if size < 1024**2 else f"{size / 1024**2:.0f} MB"
    print(f"  {ctx:>7}t  {label:>10}")

print(f"\n  10 concurrent x 4096 ctx = {per_token * 4096 * 10 / 1024**3:.1f} GB")

KV Cache -- Qwen3-0.6B
Per token: 2 x 28 x 8 x 128 x 2 = 114,688 bytes (112 KB)

   Context    KV Cache
  --------  ----------
        1t      112 KB
       64t        7 MB
      256t       28 MB
     1024t      112 MB
     4096t      448 MB

  10 concurrent x 4096 ctx = 4.4 GB


---
## (Optional) Thinking Mode

Earlier, we disabled Qwen3's thinking mode. Let's turn it on and compare, so we're going to stream the same prompt with thinking off and thinking on. With thinking on, you'll see the model generate chain of though reasoning inside these <think> tags before the actual answer, which ends up being usually better, but it uses significantly more tokens. More KV cache, more compute, and longer response time. This is the kind of tradeoff you navigate in production: better answers versus more cost per request.

In [9]:
prompt = "What makes continuous batching better than static batching?"

for label, thinking, max_tok in [
    ("Thinking OFF", False, 80), ("Thinking ON", True, 200)]:
    print(f"=== {label} ===\n")
    start = time.time()
    tokens = 0
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tok, temperature=0.7, stream=True,
        extra_body={"chat_template_kwargs": {"enable_thinking": thinking}},
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            sys.stdout.write(chunk.choices[0].delta.content)
            sys.stdout.flush()
            tokens += 1
    elapsed = time.time() - start
    print(f"\n  [{tokens} tokens, {elapsed:.2f}s]\n")

=== Thinking OFF ===

Continuous batching is generally considered more effective and efficient than static batching in several key areas:

### 1. **Improved Material Flow and Efficiency**
   - **Continuous batching** allows for real-time adjustments and optimization of material flow, reducing idle time and increasing throughput.
   - It can dynamically adjust to changes in demand or production, ensuring that the system operates at maximum capacity.

### 2. **Better
  [80 tokens, 7.60s]

=== Thinking ON ===

<think>
Okay, the user is asking why continuous batching is better than static batching. Let me start by recalling what these terms mean. Static batching typically refers to a method where the mixture is prepared in batches at fixed times, like using a mixer with a fixed time interval. Continuous batching, on the other hand, involves continuously blending the ingredients, allowing for real-time adjustments and better control.

First, I should mention the key differences. Continuous 

So, let's recap! You connected to a running vLLM server's OpenAI client, and explored logprobs and temperature to understand model behavior. You used the metrics endpoint to watch. 

In the next lesson, you'll benchmark it with GuideLLM, evaluate model quality with lm_eval